# Colab MIDI Continuation (Variant E GDN)

This notebook is self-contained for Colab and targets the Pulse88 Variant E GDN checkpoint.

It downloads model files and tokenizer from a **public** Hugging Face repo, then continues one MIDI file or a batch of MIDI files.

What you provide:
- a public Hugging Face repo id (defaults to `Chickaboo/Pulse88-E-40M-Alpha-Preview`)
- `INPUT_MIDI_SOURCE` set to either:
  - a single MIDI file path (`.mid` / `.midi`), or
  - a folder containing MIDI files

Input examples:
- single file: `/content/uploaded_midis/song.mid`
- folder: `/content/uploaded_midis`

What this notebook does:
1. Downloads the model bundle from Hugging Face (no token/login required).
2. Loads Variant E (GDN + sparse attention) and tokenizer.
3. Resolves either single-file or folder input into a sorted MIDI list.
4. Processes each song and outputs:
   - continuation MIDI
   - seed audio preview
   - continuation audio preview
   - seed-vs-continuation comparison PNG

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REQUIRED_MODULES = {
    "pretty_midi": "pretty_midi>=0.2.10",
    "matplotlib": "matplotlib>=3.7.0",
    "safetensors": "safetensors>=0.4.0",
    "huggingface_hub": "huggingface_hub>=0.24.0",
}
OPTIONAL_MODULES = {
    "fla": "flash-linear-attention",
}

missing_required = [
    spec
    for module_name, spec in REQUIRED_MODULES.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_required:
    print(f"Installing required package(s): {missing_required}")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--disable-pip-version-check",
            *missing_required,
        ],
        check=True,
    )

missing_optional = [
    spec
    for module_name, spec in OPTIONAL_MODULES.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_optional:
    print(f"Attempting optional package install for GDN kernels: {missing_optional}")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--disable-pip-version-check",
            *missing_optional,
        ],
        check=False,
    )
    still_missing_optional = [
        module_name
        for module_name in OPTIONAL_MODULES.keys()
        if importlib.util.find_spec(module_name) is None
    ]
    if still_missing_optional:
        print(
            "Optional GDN kernel package is still unavailable. "
            "The notebook will fall back to an approximation if strict GDN kernels cannot be loaded."
        )

import torch

if torch.cuda.is_available():
    try:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

PROJECT_ROOT = Path("/content/pulse88_colab")
ASSET_DIR = PROJECT_ROOT / "assets"
MODEL_DIR = PROJECT_ROOT / "models"
TOKENIZER_DIR = PROJECT_ROOT / "tokenizer"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
SEED_DIR = PROJECT_ROOT / "seed"
for path in [PROJECT_ROOT, ASSET_DIR, MODEL_DIR, TOKENIZER_DIR, OUTPUT_DIR, SEED_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
from pathlib import Path
import os

INPUT_MIDI_SOURCE = Path(os.environ.get("INPUT_MIDI_SOURCE", "/content/uploaded_midis")).expanduser()
MIDI_EXTENSIONS = {".mid", ".midi"}
RECURSIVE_SEARCH = False  # Applies only when INPUT_MIDI_SOURCE is a directory.

if not INPUT_MIDI_SOURCE.exists():
    raise FileNotFoundError(
        "Set INPUT_MIDI_SOURCE to a MIDI file or a folder containing MIDI files. "
        f"Current value: {INPUT_MIDI_SOURCE}"
    )

if INPUT_MIDI_SOURCE.is_file():
    if INPUT_MIDI_SOURCE.suffix.lower() not in MIDI_EXTENSIONS:
        raise ValueError(f"INPUT_MIDI_SOURCE is not a MIDI file: {INPUT_MIDI_SOURCE}")
    MIDI_INPUT_FILES = [INPUT_MIDI_SOURCE]
    input_mode = "file"
elif INPUT_MIDI_SOURCE.is_dir():
    input_mode = "folder"
    if RECURSIVE_SEARCH:
        candidates = [
            path
            for path in INPUT_MIDI_SOURCE.rglob("*")
            if path.is_file() and path.suffix.lower() in MIDI_EXTENSIONS
        ]
        MIDI_INPUT_FILES = sorted(
            candidates,
            key=lambda p: str(p.relative_to(INPUT_MIDI_SOURCE)).lower(),
        )
    else:
        candidates = [
            path
            for path in INPUT_MIDI_SOURCE.iterdir()
            if path.is_file() and path.suffix.lower() in MIDI_EXTENSIONS
        ]
        MIDI_INPUT_FILES = sorted(candidates, key=lambda p: p.name.lower())
else:
    raise ValueError(f"Unsupported input path type: {INPUT_MIDI_SOURCE}")

if not MIDI_INPUT_FILES:
    raise FileNotFoundError(
        f"No MIDI files found at {INPUT_MIDI_SOURCE}. Expected .mid or .midi files."
    )

print(f"Input mode: {input_mode}")
print(f"Input source: {INPUT_MIDI_SOURCE}")
print(f"Found {len(MIDI_INPUT_FILES)} MIDI file(s).")
for idx, path in enumerate(MIDI_INPUT_FILES[:10], start=1):
    print(f"  {idx:02d}. {path.name}")
if len(MIDI_INPUT_FILES) > 10:
    print(f"  ... ({len(MIDI_INPUT_FILES) - 10} more)")

In [ ]:
import os
import shutil
from pathlib import Path

import torch
from huggingface_hub import snapshot_download

HF_REPO_ID = os.environ.get("HF_REPO_ID", "Chickaboo/Pulse88-E-40M-Alpha-Preview").strip()
HF_REVISION = os.environ.get("HF_REVISION", "").strip()
if not HF_REPO_ID:
    raise ValueError("Set HF_REPO_ID to a public Hugging Face repo id.")

snapshot_kwargs = {
    "repo_id": HF_REPO_ID,
    "local_dir": str(ASSET_DIR),
    "local_dir_use_symlinks": False,
}
if HF_REVISION:
    snapshot_kwargs["revision"] = HF_REVISION

print(f"HF repo: {HF_REPO_ID}")
if HF_REVISION:
    print(f"HF revision: {HF_REVISION}")
ASSET_ROOT = Path(snapshot_download(**snapshot_kwargs))
print(f"Downloaded assets to: {ASSET_ROOT}")


def _bundle_score(model_path: Path, state_path: Path | None, tokenizer_path: Path | None) -> int:
    score = 0
    model_name = model_path.name.lower()
    if "latest" in model_name:
        score += 120
    elif "best" in model_name:
        score += 100
    elif "checkpoint" in model_name or "epoch" in model_name:
        score += 60
    elif "model" in model_name:
        score += 30
    score += min(25, int(model_path.stat().st_size // 1_000_000))

    if state_path is not None:
        state_name = state_path.name.lower()
        if "latest" in state_name:
            score += 80
        elif "best" in state_name:
            score += 70
        elif "state" in state_name:
            score += 40

    if tokenizer_path is not None:
        score += 60 if tokenizer_path.name == "custom_tokenizer.json" else 30

    return score


def _select_best_bundle(asset_root: Path) -> tuple[dict, list[dict]]:
    model_paths = sorted(path for path in asset_root.rglob("*.safetensors") if path.is_file())
    if not model_paths:
        raise FileNotFoundError(
            "No .safetensors files were found in the downloaded Hugging Face snapshot."
        )

    bundles = []
    for model_path in model_paths:
        parent = model_path.parent
        stem = model_path.stem

        state_candidates = [
            parent / f"{stem}_state.pt",
            parent / f"{stem.replace('_model', '')}_state.pt" if stem.endswith("_model") else parent / f"{stem}_state.pt",
            parent / "latest_state.pt",
            parent / "best_state.pt",
        ]
        state_candidates.extend(sorted(parent.glob("*_state.pt")))
        state_path = next((candidate for candidate in state_candidates if candidate.exists()), None)

        tokenizer_candidates = [
            parent / "custom_tokenizer.json",
            parent / "tokenizer.json",
            parent.parent / "custom_tokenizer.json",
            parent.parent / "tokenizer.json",
        ]
        tokenizer_path = next((candidate for candidate in tokenizer_candidates if candidate.exists()), None)
        if tokenizer_path is None:
            continue

        try:
            state_payload = torch.load(state_path, map_location="cpu") if state_path is not None else {}
        except Exception:
            state_payload = {}
        if not isinstance(state_payload, dict):
            state_payload = {}

        model_payload = dict(state_payload.get("model_config") or {})
        data_payload = dict(state_payload.get("data_config") or {})

        score = _bundle_score(model_path, state_path, tokenizer_path)
        if str(data_payload.get("tokenization_strategy", "")).strip().lower() == "custom_delta":
            score += 40
        if int(data_payload.get("vocab_size", 0) or 0) == 171:
            score += 10
        if int(model_payload.get("d_model", 0) or 0) > 0:
            score += 10
        if int(model_payload.get("n_layers", 0) or 0) > 0:
            score += 10

        bundles.append(
            {
                "score": int(score),
                "model_path": model_path,
                "state_path": state_path,
                "tokenizer_path": tokenizer_path,
                "summary": {
                    "tokenization_strategy": str(data_payload.get("tokenization_strategy", "")),
                    "d_model": int(model_payload.get("d_model", 0) or 0),
                    "n_layers": int(model_payload.get("n_layers", 0) or 0),
                    "vocab_size": int(data_payload.get("vocab_size", 0) or 0),
                },
            }
        )

    if not bundles:
        raise FileNotFoundError(
            "Could not find a complete model/state/tokenizer bundle in the Hugging Face snapshot."
        )

    bundles.sort(key=lambda item: item["score"], reverse=True)
    return bundles[0], bundles[: min(5, len(bundles))]


selected, top_candidates = _select_best_bundle(ASSET_ROOT)
selected_model_path = selected["model_path"]
selected_state_path = selected["state_path"]
selected_tokenizer_path = selected["tokenizer_path"]

if selected_state_path is None:
    raise FileNotFoundError(
        "Could not find a matching sidecar state file for the selected model bundle."
    )

shutil.copy2(selected_model_path, MODEL_DIR / "latest.safetensors")
shutil.copy2(selected_state_path, MODEL_DIR / "latest_state.pt")
if selected_tokenizer_path.name == "custom_tokenizer.json":
    shutil.copy2(selected_tokenizer_path, TOKENIZER_DIR / "custom_tokenizer.json")
else:
    shutil.copy2(selected_tokenizer_path, TOKENIZER_DIR / "tokenizer.json")

print("Selected asset bundle:")
print(f"  model: {selected_model_path}")
print(f"  state: {selected_state_path}")
print(f"  tokenizer: {selected_tokenizer_path}")
print(f"  score: {selected['score']}")
print(f"  summary: {selected['summary']}")

if len(top_candidates) > 1:
    print("Top candidate scores:")
    for item in top_candidates:
        model_rel = item["model_path"].relative_to(ASSET_ROOT)
        state_rel = item["state_path"].relative_to(ASSET_ROOT) if item["state_path"] else "missing"
        tok_rel = item["tokenizer_path"].relative_to(ASSET_ROOT) if item["tokenizer_path"] else "missing"
        print(f"  score={item['score']:3d} model={model_rel} state={state_rel} tokenizer={tok_rel}")

print("Available model files:")
for path in sorted(ASSET_ROOT.rglob("*.safetensors")):
    print(f"  {path.relative_to(ASSET_ROOT)}")
print("Available tokenizer files:")
for path in sorted(list(ASSET_ROOT.rglob("custom_tokenizer.json")) + list(ASSET_ROOT.rglob("tokenizer.json"))):
    print(f"  {path.relative_to(ASSET_ROOT)}")

In [ ]:
import json
import math
import wave
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any, List, Optional, Sequence, Tuple

import numpy as np
import pretty_midi

try:
    import matplotlib.pyplot as plt
except Exception as exc:  # pragma: no cover - optional dependency
    plt = None  # type: ignore
    warnings.warn(f"matplotlib import failed. Visualization will be disabled. Details: {exc}")


def midi_duration(midi_path: str | Path) -> float:
    """Return MIDI duration in seconds."""

    midi = pretty_midi.PrettyMIDI(str(midi_path))
    return float(midi.get_end_time())


def render_midi_audio(
    midi_path: str | Path,
    wav_path: str | Path,
    *,
    sample_rate: int = 22050,
) -> Path:
    """Render a MIDI file to a small WAV preview using pretty_midi's synth."""

    midi = pretty_midi.PrettyMIDI(str(midi_path))
    try:
        waveform = midi.synthesize(fs=int(sample_rate))
    except Exception:
        duration = max(0.5, float(midi.get_end_time()))
        waveform = np.zeros(int(duration * int(sample_rate)), dtype=np.float32)

    waveform = np.asarray(waveform, dtype=np.float32)
    if waveform.ndim == 2:
        waveform = waveform.mean(axis=1)
    if waveform.size == 0:
        waveform = np.zeros(int(sample_rate * 0.5), dtype=np.float32)

    peak = float(np.max(np.abs(waveform))) if waveform.size else 0.0
    if peak > 0:
        waveform = 0.95 * waveform / peak

    pcm16 = np.clip(waveform, -1.0, 1.0)
    pcm16 = (pcm16 * 32767.0).astype(np.int16)

    out_path = Path(wav_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with wave.open(str(out_path), "wb") as wav_file:
        wav_file.setnchannels(1)
        wav_file.setsampwidth(2)
        wav_file.setframerate(int(sample_rate))
        wav_file.writeframes(pcm16.tobytes())
    return out_path


def _extract_note_events(midi_path: str | Path) -> List[Tuple[float, float, int, int]]:
    """Extract note events `(start, end, pitch, velocity)` for piano range."""

    midi = pretty_midi.PrettyMIDI(str(midi_path))
    events: List[Tuple[float, float, int, int]] = []
    for inst in midi.instruments:
        if inst.is_drum:
            continue
        for n in inst.notes:
            if 21 <= n.pitch <= 108:
                events.append((n.start, n.end, n.pitch, n.velocity))
    return events


def visualize_pianoroll(
    midi_path: str | Path,
    title: str = "",
    save_path: Optional[str | Path] = None,
) -> None:
    """Render a pianoroll plot for one MIDI file."""

    if plt is None:  # pragma: no cover - optional dependency
        warnings.warn("visualize_pianoroll called but matplotlib is not installed; skipping visualization")
        return

    events = _extract_note_events(midi_path)
    cmap = plt.get_cmap("viridis")

    fig, ax = plt.subplots(figsize=(14, 5))
    for start, end, pitch, velocity in events:
        color = cmap(velocity / 127.0)
        ax.hlines(y=pitch, xmin=start, xmax=end, linewidth=2.0, color=color)

    ax.set_xlabel("Time (s)")
    ax.set_ylabel("MIDI Pitch")
    ax.set_ylim(21, 108)
    ax.set_title(title or f"Piano Roll: {Path(midi_path).name}")
    ax.grid(alpha=0.2)
    fig.tight_layout()

    if save_path is None:
        plt.show()
    else:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150)
        plt.close(fig)


def compare_pianorolls(
    seed_path: str | Path,
    continuation_path: str | Path,
    save_path: Optional[str | Path] = None,
) -> None:
    """Render side-by-side timeline comparison for seed and continuation."""

    if plt is None:  # pragma: no cover - optional dependency
        warnings.warn("compare_pianorolls called but matplotlib is not installed; skipping visualization")
        return

    seed_events = _extract_note_events(seed_path)
    cont_events = _extract_note_events(continuation_path)
    seed_cmap = plt.get_cmap("Blues")
    cont_cmap = plt.get_cmap("Oranges")

    seed_dur = midi_duration(seed_path)

    fig, ax = plt.subplots(figsize=(16, 5))

    for start, end, pitch, velocity in seed_events:
        color = seed_cmap(0.3 + 0.7 * velocity / 127.0)
        ax.hlines(y=pitch, xmin=start, xmax=end, linewidth=2.0, color=color)

    for start, end, pitch, velocity in cont_events:
        start += seed_dur
        end += seed_dur
        color = cont_cmap(0.3 + 0.7 * velocity / 127.0)
        ax.hlines(y=pitch, xmin=start, xmax=end, linewidth=2.0, color=color)

    ax.axvline(seed_dur, linestyle="--", linewidth=2, color="black", alpha=0.7)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("MIDI Pitch")
    ax.set_ylim(21, 108)
    ax.set_title("Seed | Continuation")
    ax.grid(alpha=0.2)
    fig.tight_layout()

    if save_path is None:
        plt.show()
    else:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150)
        plt.close(fig)


@dataclass(frozen=True)
class _TokenSpec:
    delta_start: int = 0
    delta_end: int = 31
    pitch_start: int = 32
    pitch_end: int = 119
    duration_start: int = 120
    duration_end: int = 151
    velocity_start: int = 152
    velocity_end: int = 167
    pad_id: int = 168
    bos_id: int = 169
    eos_id: int = 170
    event_size: int = 4

    @property
    def vocab_size(self) -> int:
        return 171


class CustomDeltaTokenizer:
    """Quad tokenizer for solo piano: [delta_onset, pitch, duration, velocity_bin]."""

    def __init__(
        self,
        *,
        default_velocity: int = 88,
        include_special_tokens: bool = False,
    ) -> None:
        self.spec = _TokenSpec()
        self.default_velocity = int(max(1, min(127, default_velocity)))
        self.include_special_tokens = bool(include_special_tokens)

        self._delta_bins = np.concatenate(
            [
                np.asarray([0.0], dtype=np.float64),
                np.logspace(math.log10(1e-4), math.log10(2.0), num=31),
            ],
            axis=0,
        )
        self._duration_bins = np.logspace(
            math.log10(0.05),
            math.log10(4.0),
            num=32,
        ).astype(np.float64)

    @property
    def vocab_size(self) -> int:
        return self.spec.vocab_size

    @property
    def event_size(self) -> int:
        return int(self.spec.event_size)

    @property
    def pad_id(self) -> int:
        return self.spec.pad_id

    @property
    def bos_id(self) -> int:
        return self.spec.bos_id

    @property
    def eos_id(self) -> int:
        return self.spec.eos_id

    def save(self, path: str) -> None:
        out_path = Path(path)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        payload = {
            "type": "CustomDeltaTokenizer",
            "version": 1,
            "vocab_size": int(self.vocab_size),
            "default_velocity": int(self.default_velocity),
            "event_size": int(self.event_size),
            "include_special_tokens": bool(self.include_special_tokens),
            "token_ids": {
                "delta": [self.spec.delta_start, self.spec.delta_end],
                "pitch": [self.spec.pitch_start, self.spec.pitch_end],
                "duration": [self.spec.duration_start, self.spec.duration_end],
                "velocity": [self.spec.velocity_start, self.spec.velocity_end],
                "pad": self.spec.pad_id,
                "bos": self.spec.bos_id,
                "eos": self.spec.eos_id,
            },
        }
        out_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

    @classmethod
    def load(cls, path: str) -> "CustomDeltaTokenizer":
        in_path = Path(path)
        if not in_path.exists():
            raise FileNotFoundError(f"Tokenizer file not found: {in_path}")

        payload = json.loads(in_path.read_text(encoding="utf-8"))
        if str(payload.get("type", "")) != "CustomDeltaTokenizer":
            raise ValueError("Unsupported tokenizer payload. Expected type='CustomDeltaTokenizer'.")

        return cls(
            default_velocity=int(payload.get("default_velocity", 88)),
            include_special_tokens=bool(payload.get("include_special_tokens", False)),
        )

    def _note_events(self, midi_path: Path) -> List[Tuple[float, int, float, int]]:
        midi = pretty_midi.PrettyMIDI(str(midi_path))
        events: List[Tuple[float, int, float, int]] = []

        for inst in midi.instruments:
            if inst.is_drum:
                continue
            for note in inst.notes:
                onset = float(max(0.0, note.start))
                duration = float(max(1e-4, note.end - note.start))
                pitch = int(note.pitch)
                velocity = int(max(0, min(127, int(note.velocity))))
                if pitch < 21 or pitch > 108:
                    continue
                events.append((onset, pitch, duration, velocity))

        events.sort(key=lambda x: (x[0], x[1], x[2], x[3]))
        return events

    @staticmethod
    def _nearest_bin(value: float, bins: np.ndarray) -> int:
        idx = int(np.argmin(np.abs(bins - float(value))))
        return idx

    def _quantize_delta(self, delta_seconds: float) -> int:
        clamped = float(max(0.0, min(2.0, delta_seconds)))
        idx = self._nearest_bin(clamped, self._delta_bins)
        return int(self.spec.delta_start + idx)

    def _quantize_duration(self, duration_seconds: float) -> int:
        clamped = float(max(0.05, min(4.0, duration_seconds)))
        idx = self._nearest_bin(clamped, self._duration_bins)
        return int(self.spec.duration_start + idx)

    def _quantize_pitch(self, pitch: int) -> int:
        pitch_i = int(max(21, min(108, pitch)))
        return int(self.spec.pitch_start + (pitch_i - 21))

    def _quantize_velocity(self, velocity: int) -> int:
        vel = int(max(0, min(127, int(velocity))))
        bin_idx = int(round((float(vel) / 127.0) * 15.0))
        bin_idx = max(0, min(15, bin_idx))
        return int(self.spec.velocity_start + bin_idx)

    def _dequantize_delta(self, token_id: int) -> float:
        idx = int(token_id) - self.spec.delta_start
        idx = max(0, min(31, idx))
        return float(self._delta_bins[idx])

    def _dequantize_duration(self, token_id: int) -> float:
        idx = int(token_id) - self.spec.duration_start
        idx = max(0, min(31, idx))
        return float(self._duration_bins[idx])

    def _dequantize_pitch(self, token_id: int) -> int:
        idx = int(token_id) - self.spec.pitch_start
        idx = max(0, min(87, idx))
        return int(21 + idx)

    def _dequantize_velocity(self, token_id: int) -> int:
        idx = int(token_id) - self.spec.velocity_start
        idx = max(0, min(15, idx))
        return int(round((float(idx) / 15.0) * 127.0))

    def _encode_events(self, midi_path: Path) -> Tuple[List[int], List[float], List[float]]:
        events = self._note_events(midi_path)

        token_ids: List[int] = []
        onset_times: List[float] = []
        durations: List[float] = []
        prev_onset = 0.0

        if self.include_special_tokens:
            token_ids.append(self.spec.bos_id)
            onset_times.append(0.0)
            durations.append(1e-4)

        for onset, pitch, duration, velocity in events:
            delta = float(max(0.0, onset - prev_onset))
            prev_onset = onset

            d_tok = self._quantize_delta(delta)
            p_tok = self._quantize_pitch(pitch)
            u_tok = self._quantize_duration(duration)
            v_tok = self._quantize_velocity(velocity)
            token_ids.extend([d_tok, p_tok, u_tok, v_tok])
            onset_times.extend([float(onset), float(onset), float(onset), float(onset)])
            durations.extend([float(duration), float(duration), float(duration), float(duration)])

        if self.include_special_tokens:
            end_onset = float(onset_times[-1]) if onset_times else 0.0
            token_ids.append(self.spec.eos_id)
            onset_times.append(end_onset)
            durations.append(1e-4)

        if len(token_ids) != len(onset_times):
            raise AssertionError(
                "CustomDeltaTokenizer alignment error: "
                f"len(ids)={len(token_ids)} len(onsets)={len(onset_times)}"
            )
        return token_ids, onset_times, durations

    def encode(self, midi_path: Path) -> List[int]:
        token_ids, _, _ = self._encode_events(Path(midi_path))
        return token_ids

    def encode_with_time_features(self, midi_path: Path) -> Tuple[List[int], List[float], List[float]]:
        return self._encode_events(Path(midi_path))

    def _is_delta(self, token_id: int) -> bool:
        return self.spec.delta_start <= int(token_id) <= self.spec.delta_end

    def _is_pitch(self, token_id: int) -> bool:
        return self.spec.pitch_start <= int(token_id) <= self.spec.pitch_end

    def _is_duration(self, token_id: int) -> bool:
        return self.spec.duration_start <= int(token_id) <= self.spec.duration_end

    def _is_velocity(self, token_id: int) -> bool:
        return self.spec.velocity_start <= int(token_id) <= self.spec.velocity_end

    def _is_special(self, token_id: int) -> bool:
        token = int(token_id)
        return token in {self.spec.pad_id, self.spec.bos_id, self.spec.eos_id}

    def decode_token_id_events(self, token_id: int) -> List[str]:
        token = int(token_id)
        if self._is_delta(token):
            return [f"Delta_{self._dequantize_delta(token):.6f}"]
        if self._is_pitch(token):
            return [f"Pitch_{self._dequantize_pitch(token)}"]
        if self._is_duration(token):
            return [f"Duration_{self._dequantize_duration(token):.6f}"]
        if self._is_velocity(token):
            return [f"Velocity_{self._dequantize_velocity(token)}"]
        if token == self.spec.pad_id:
            return ["PAD_None"]
        if token == self.spec.bos_id:
            return ["BOS_None"]
        if token == self.spec.eos_id:
            return ["EOS_None"]
        return []

    def decode(self, token_ids: Sequence[int], output_path: Path | str | None = None) -> Any:
        midi = pretty_midi.PrettyMIDI()
        piano = pretty_midi.Instrument(program=0)

        tokens = [int(t) for t in token_ids]
        i = 0
        onset = 0.0

        while i < len(tokens):
            tok = tokens[i]
            if tok == self.spec.eos_id:
                break
            if self._is_special(tok):
                i += 1
                continue
            if not self._is_delta(tok):
                i += 1
                continue
            if i + 3 >= len(tokens):
                break

            p_tok = tokens[i + 1]
            d_tok = tokens[i + 2]
            v_tok = tokens[i + 3]
            if (not self._is_pitch(p_tok)) or (not self._is_duration(d_tok)) or (not self._is_velocity(v_tok)):
                i += 1
                continue

            delta = self._dequantize_delta(tok)
            pitch = self._dequantize_pitch(p_tok)
            duration = self._dequantize_duration(d_tok)
            velocity = self._dequantize_velocity(v_tok)

            onset = float(max(0.0, onset + max(0.0, delta)))
            end = float(max(onset + 1e-4, onset + duration))
            note = pretty_midi.Note(
                velocity=int(velocity),
                pitch=int(pitch),
                start=float(onset),
                end=float(end),
            )
            piano.notes.append(note)
            i += 4

        midi.instruments.append(piano)

        if output_path is not None:
            out_path = Path(output_path)
            out_path.parent.mkdir(parents=True, exist_ok=True)
            midi.write(str(out_path))

        return midi

    def verify_roundtrip(self, midi_path: Path) -> bool:
        try:
            ids = self.encode(Path(midi_path))
            _ = self.decode(ids)
            return len(ids) > 0
        except Exception:
            return False

In [ ]:
import json
import math
import os
import re
import time
import importlib
from dataclasses import dataclass, fields
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from safetensors import safe_open as safetensors_safe_open
from safetensors.torch import load_file as safetensors_load_file

_ipd = None
try:
    _ipd = importlib.import_module("IPython.display")
except Exception:
    _ipd = None
Audio = getattr(_ipd, "Audio", None)
Image = getattr(_ipd, "Image", None)
display = getattr(_ipd, "display", None)


def _strip_dataparallel_prefix(state_dict: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    out: Dict[str, torch.Tensor] = {}
    for key, value in state_dict.items():
        if key.startswith("module."):
            out[key[len("module."):]] = value
        else:
            out[key] = value
    return out


def _rotate_half(x: torch.Tensor) -> torch.Tensor:
    half = x.shape[-1] // 2
    x1 = x[..., :half]
    x2 = x[..., half:]
    return torch.cat([-x2, x1], dim=-1)


class RotaryEmbedding(nn.Module):
    def __init__(self, dim: int, base: float = 10000.0) -> None:
        super().__init__()
        if dim <= 0 or dim % 2 != 0:
            raise ValueError("RoPE dimension must be positive and even")
        self.dim = int(dim)
        self.base = float(base)
        self._seq_len_cached = 0
        self.register_buffer("_cos_cached", torch.empty(0), persistent=False)
        self.register_buffer("_sin_cached", torch.empty(0), persistent=False)

    def _build_cache(self, seq_len: int, device: torch.device, dtype: torch.dtype) -> None:
        inv_freq = 1.0 / (
            self.base
            ** (
                torch.arange(0, self.dim, 2, device=device, dtype=torch.float32)
                / float(self.dim)
            )
        )
        positions = torch.arange(seq_len, device=device, dtype=torch.float32)
        freqs = torch.outer(positions, inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self._cos_cached = torch.cos(emb).to(dtype=dtype)
        self._sin_cached = torch.sin(emb).to(dtype=dtype)
        self._seq_len_cached = int(seq_len)

    def _get_cos_sin(self, seq_len: int, device: torch.device, dtype: torch.dtype, offset: int = 0) -> Tuple[torch.Tensor, torch.Tensor]:
        required = int(seq_len + max(0, int(offset)))
        if (
            self._cos_cached.numel() == 0
            or self._sin_cached.numel() == 0
            or self._seq_len_cached < required
            or self._cos_cached.device != device
            or self._cos_cached.dtype != dtype
        ):
            self._build_cache(required, device=device, dtype=dtype)
        start = int(max(0, offset))
        end = start + int(seq_len)
        return self._cos_cached[start:end], self._sin_cached[start:end]

    def apply(self, q: torch.Tensor, k: torch.Tensor, offset: int = 0) -> Tuple[torch.Tensor, torch.Tensor]:
        seq_len = int(q.shape[-2])
        cos, sin = self._get_cos_sin(seq_len=seq_len, device=q.device, dtype=q.dtype, offset=int(offset))
        cos = cos.view(1, 1, seq_len, self.dim)
        sin = sin.view(1, 1, seq_len, self.dim)
        return (q * cos) + (_rotate_half(q) * sin), (k * cos) + (_rotate_half(k) * sin)


class GQABlock(nn.Module):
    def __init__(
        self,
        d_model: int,
        num_heads: int,
        num_kv_heads: Optional[int] = None,
        dropout: float = 0.1,
        rope_base: float = 10000.0,
    ) -> None:
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError("d_model must be divisible by num_heads")
        self.d_model = int(d_model)
        self.num_heads = int(num_heads)
        self.head_dim = self.d_model // self.num_heads
        if self.head_dim % 2 != 0:
            raise ValueError("head_dim must be even for RoPE")

        kv = int(num_kv_heads) if num_kv_heads is not None else int(num_heads)
        kv = max(1, kv)
        if self.num_heads % kv != 0:
            raise ValueError("num_heads must be divisible by num_kv_heads")
        self.num_kv_heads = int(kv)
        self.group_size = self.num_heads // self.num_kv_heads

        self.q_proj = nn.Linear(self.d_model, self.num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(self.d_model, self.num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(self.d_model, self.num_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(self.d_model, self.d_model, bias=False)
        self.out_dropout = nn.Dropout(float(dropout))
        self.rope = RotaryEmbedding(dim=self.head_dim, base=float(rope_base))

    def forward(self, x: torch.Tensor, position_offset: int = 0) -> torch.Tensor:
        batch_size, seq_len, _ = x.shape
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.num_kv_heads, self.head_dim).transpose(1, 2)

        if self.group_size > 1:
            k = k.repeat_interleave(self.group_size, dim=1)
            v = v.repeat_interleave(self.group_size, dim=1)

        q, k = self.rope.apply(q, k, offset=int(max(0, position_offset)))
        attn_out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=self.out_dropout.p if self.training else 0.0,
            is_causal=True,
        )
        out = attn_out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.out_dropout(self.out_proj(out))


@dataclass
class SamplingDiagnostics:
    raw_top1_prob: torch.Tensor
    final_top1_prob: torch.Tensor
    candidate_count: torch.Tensor


def _apply_repetition_penalty(
    logits: torch.Tensor,
    context_tokens: torch.Tensor,
    repetition_penalty: float,
    recent_window: int,
) -> torch.Tensor:
    if repetition_penalty <= 1.0 or recent_window <= 0:
        return logits
    adjusted = logits.clone()
    recent = context_tokens[:, -min(recent_window, context_tokens.shape[1]):]
    for batch_idx in range(adjusted.shape[0]):
        token_ids = torch.unique(recent[batch_idx])
        token_logits = adjusted[batch_idx, token_ids]
        adjusted[batch_idx, token_ids] = torch.where(
            token_logits < 0,
            token_logits * repetition_penalty,
            token_logits / repetition_penalty,
        )
    return adjusted


def _apply_topk_topp_filter(
    logits: torch.Tensor,
    top_k: Optional[int],
    top_p: Optional[float],
    min_tokens_to_keep: int,
) -> torch.Tensor:
    batch_size, vocab_size = logits.shape
    keep_k = vocab_size
    if top_k is not None and top_k > 0:
        keep_k = min(max(int(top_k), int(min_tokens_to_keep)), vocab_size)

    topk_indices = torch.topk(logits, k=keep_k, dim=-1).indices
    candidate_mask = torch.zeros_like(logits, dtype=torch.bool)
    candidate_mask.scatter_(dim=-1, index=topk_indices, value=True)

    if top_p is not None and 0.0 < top_p < 1.0:
        sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
        sorted_probs = torch.softmax(sorted_logits, dim=-1)
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
        remove_mask = cumulative_probs > float(top_p)
        if min_tokens_to_keep > 0:
            remove_mask[..., : int(min_tokens_to_keep)] = False
        nucleus_keep = ~remove_mask
        nucleus_mask = torch.zeros_like(candidate_mask)
        nucleus_mask.scatter_(dim=-1, index=sorted_indices, src=nucleus_keep)
        candidate_mask = candidate_mask & nucleus_mask

    counts = candidate_mask.sum(dim=-1)
    if bool((counts < int(min_tokens_to_keep)).any()):
        top_idx = torch.topk(logits, k=int(min_tokens_to_keep), dim=-1).indices
        for row_idx in range(batch_size):
            if int(counts[row_idx].item()) < int(min_tokens_to_keep):
                candidate_mask[row_idx, top_idx[row_idx]] = True

    return logits.masked_fill(~candidate_mask, float("-inf"))


def sample_next_token(
    logits: torch.Tensor,
    context_tokens: torch.Tensor,
    temperature: float,
    top_p: Optional[float],
    top_k: Optional[int],
    repetition_penalty: float,
    recent_window: int,
    min_tokens_to_keep: int,
) -> Tuple[torch.Tensor, SamplingDiagnostics]:
    temperature = max(float(temperature), 0.1)
    min_tokens_to_keep = max(1, int(min_tokens_to_keep))

    penalized = _apply_repetition_penalty(
        logits=logits,
        context_tokens=context_tokens,
        repetition_penalty=float(repetition_penalty),
        recent_window=int(recent_window),
    )
    scaled = penalized / temperature
    raw_probs = torch.softmax(scaled, dim=-1)
    raw_top1_prob = raw_probs.max(dim=-1).values

    filtered_logits = _apply_topk_topp_filter(
        logits=scaled,
        top_k=top_k,
        top_p=top_p,
        min_tokens_to_keep=min_tokens_to_keep,
    )
    candidate_mask = torch.isfinite(filtered_logits)
    probs = torch.softmax(filtered_logits, dim=-1)
    probs = probs / probs.sum(dim=-1, keepdim=True).clamp_min(1e-12)

    diagnostics = SamplingDiagnostics(
        raw_top1_prob=raw_top1_prob,
        final_top1_prob=probs.max(dim=-1).values,
        candidate_count=candidate_mask.sum(dim=-1),
    )
    next_token = torch.multinomial(probs, num_samples=1)
    return next_token, diagnostics


GDN_AVAILABLE = False
_GatedDeltaNet = None


def _try_import_fla() -> bool:
    global GDN_AVAILABLE, _GatedDeltaNet
    try:
        fla_layers = importlib.import_module("fla.layers")
        _GDN = getattr(fla_layers, "GatedDeltaNet")

        _GatedDeltaNet = _GDN
        GDN_AVAILABLE = True
        return True
    except Exception:
        _GatedDeltaNet = None
        GDN_AVAILABLE = False
        return False


_try_import_fla()


class _GatedDeltaFallback(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1) -> None:
        super().__init__()
        self.mix = nn.Linear(d_model, d_model * 2, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(float(dropout))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        u, g = self.mix(x).chunk(2, dim=-1)
        y = F.silu(u) * torch.sigmoid(g)
        return self.dropout(self.out(y))


class GatedDeltaNetBlock(nn.Module):
    def __init__(
        self,
        d_model: int,
        inner_dim: int = 320,
        num_heads: int = 5,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        if inner_dim % num_heads != 0:
            raise ValueError("inner_dim must be divisible by num_heads")
        self.d_model = int(d_model)
        self.inner_dim = int(inner_dim)
        self.num_heads = int(num_heads)
        self.head_dim = self.inner_dim // self.num_heads

        self.in_proj = nn.Identity() if self.inner_dim == self.d_model else nn.Linear(self.d_model, self.inner_dim, bias=False)
        self.out_proj = nn.Identity() if self.inner_dim == self.d_model else nn.Linear(self.inner_dim, self.d_model, bias=False)

        if GDN_AVAILABLE and _GatedDeltaNet is not None:
            self.core = _GatedDeltaNet(
                hidden_size=self.inner_dim,
                num_heads=self.num_heads,
                head_dim=self.head_dim,
                mode="chunk",
                use_short_conv=True,
            )
            self.using_fallback = False
        else:
            self.core = _GatedDeltaFallback(self.inner_dim, dropout=dropout)
            self.using_fallback = True

        self.post_dropout = nn.Dropout(float(dropout))

    def _run_core(self, x: torch.Tensor) -> torch.Tensor:
        if self.using_fallback:
            return self.core(x)
        out = self.core(x)
        return out[0] if isinstance(out, tuple) else out

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.in_proj(x)
        y = self._run_core(y)
        y = self.out_proj(y)
        return self.post_dropout(y)


@dataclass
class VariantEConfig:
    vocab_size: int = 171
    d_model: int = 512
    n_layers: int = 6
    max_sequence_length: int = 1024
    dropout: float = 0.1
    attention_dropout: float = 0.1
    tie_embeddings: bool = True
    embedding_init_std: float = 0.02
    output_logit_scale: Optional[float] = None
    gdn_inner_dim: int = 256
    gdn_num_heads: int = 4
    gqa_num_heads: int = 8
    gqa_groups: int = 4
    attention_every_n_layers: int = 2
    use_v2_architecture: bool = True


class _VariantEBlock(nn.Module):
    def __init__(self, cfg: VariantEConfig, use_attention: bool) -> None:
        super().__init__()
        d = int(cfg.d_model)

        self.norm_gdn1 = nn.LayerNorm(d)
        self.gdn1 = GatedDeltaNetBlock(
            d_model=d,
            inner_dim=int(cfg.gdn_inner_dim),
            num_heads=int(cfg.gdn_num_heads),
            dropout=float(cfg.dropout),
        )

        self.norm_gdn2 = nn.LayerNorm(d)
        self.gdn2 = GatedDeltaNetBlock(
            d_model=d,
            inner_dim=int(cfg.gdn_inner_dim),
            num_heads=int(cfg.gdn_num_heads),
            dropout=float(cfg.dropout),
        )

        self.use_attention = bool(use_attention)
        if self.use_attention:
            kv_heads = max(1, int(cfg.gqa_num_heads) // max(1, int(cfg.gqa_groups)))
            self.norm_gqa = nn.LayerNorm(d)
            self.gqa = GQABlock(
                d_model=d,
                num_heads=int(cfg.gqa_num_heads),
                num_kv_heads=kv_heads,
                dropout=float(cfg.attention_dropout),
            )
        else:
            self.norm_gqa = None
            self.gqa = None

    def forward(self, x: torch.Tensor, position_offset: int) -> torch.Tensor:
        x = x + self.gdn1(self.norm_gdn1(x))
        x = x + self.gdn2(self.norm_gdn2(x))
        if self.use_attention and self.gqa is not None and self.norm_gqa is not None:
            x = x + self.gqa(self.norm_gqa(x), position_offset=int(max(0, position_offset)))
        return x


class VariantEModel(nn.Module):
    def __init__(self, config: Optional[VariantEConfig] = None) -> None:
        super().__init__()
        self.config = config or VariantEConfig()
        cfg = self.config

        self.vocab_size = int(cfg.vocab_size)
        self.d_model = int(cfg.d_model)
        self.max_sequence_length = int(cfg.max_sequence_length)

        self.token_embedding = nn.Embedding(self.vocab_size, self.d_model)
        self.position_embedding = nn.Embedding(self.max_sequence_length, self.d_model)
        self.dropout = nn.Dropout(float(cfg.dropout))

        n_layers = int(cfg.n_layers)
        attn_stride = max(1, int(cfg.attention_every_n_layers))

        def _use_attention(layer_index: int) -> bool:
            is_last = layer_index == (n_layers - 1)
            return is_last or ((layer_index + 1) % attn_stride == 0)

        self.layers = nn.ModuleList(
            [_VariantEBlock(cfg, use_attention=_use_attention(i)) for i in range(n_layers)]
        )

        self.final_norm = nn.LayerNorm(self.d_model)
        self.lm_head = nn.Linear(self.d_model, self.vocab_size, bias=False)
        if bool(cfg.tie_embeddings):
            self.lm_head.weight = self.token_embedding.weight

        self.output_logit_scale = (
            1.0 / math.sqrt(float(self.d_model))
            if cfg.output_logit_scale is None
            else float(cfg.output_logit_scale)
        )

        self.last_generation_stats: Dict[str, Any] = {}

    @staticmethod
    def _to_seed_tensor(seed_tokens: Sequence[int] | torch.Tensor, *, device: torch.device) -> torch.Tensor:
        if isinstance(seed_tokens, torch.Tensor):
            if seed_tokens.ndim == 1:
                seed = seed_tokens.unsqueeze(0)
            elif seed_tokens.ndim == 2 and int(seed_tokens.shape[0]) == 1:
                seed = seed_tokens
            else:
                raise ValueError("seed tensor must be shape (seq,) or (1, seq)")
            return seed.to(device=device, dtype=torch.long)
        vals = [int(t) for t in seed_tokens]
        if not vals:
            raise ValueError("seed_tokens cannot be empty")
        return torch.tensor(vals, dtype=torch.long, device=device).unsqueeze(0)

    @staticmethod
    def _triplet_slot(index: int) -> int:
        return int(index % 4)

    @staticmethod
    def _allowed_ids_for_slot(slot: int, vocab_size: int) -> torch.Tensor:
        if slot == 0:
            return torch.arange(0, 32, dtype=torch.long)
        if slot == 1:
            return torch.arange(32, 120, dtype=torch.long)
        if slot == 2:
            return torch.arange(120, 152, dtype=torch.long)
        if slot == 3:
            return torch.arange(152, 168, dtype=torch.long)
        return torch.arange(0, vocab_size, dtype=torch.long)

    def _mask_logits_to_triplet_slot(self, logits: torch.Tensor, slot: int) -> torch.Tensor:
        mask = torch.full_like(logits, fill_value=-float("inf"))
        allowed = self._allowed_ids_for_slot(slot, logits.shape[-1]).to(logits.device)
        mask[:, allowed] = logits[:, allowed]
        return mask

    @staticmethod
    def _delta_from_token_events(token_id: int, token_id_to_events: Any, default_step: float) -> float:
        if callable(token_id_to_events):
            try:
                events = token_id_to_events(int(token_id))
                if isinstance(events, str):
                    events = [events]
                if isinstance(events, (list, tuple)):
                    for ev in events:
                        text = str(ev)
                        if text.startswith("Delta_"):
                            return float(max(1e-4, float(text.split("_", 1)[1])))
            except Exception:
                pass
        return float(max(1e-4, default_step))

    def forward(
        self,
        token_ids: torch.Tensor,
        onset_times: torch.Tensor,
        durations: Optional[torch.Tensor] = None,
        memory: Optional[Any] = None,
        return_memory: bool = False,
        position_offset: int = 0,
    ) -> Tuple[torch.Tensor, Optional[Any]] | torch.Tensor:
        del memory
        if token_ids.ndim != 2 or onset_times.ndim != 2:
            raise ValueError("token_ids and onset_times must be rank-2")
        if token_ids.shape != onset_times.shape:
            raise ValueError("token_ids and onset_times must have same shape")
        if durations is not None and durations.shape != token_ids.shape:
            raise ValueError("durations must match token_ids shape")

        bsz, seq_len = token_ids.shape
        positions = torch.arange(
            int(max(0, position_offset)),
            int(max(0, position_offset)) + int(seq_len),
            device=token_ids.device,
        )
        positions = torch.clamp(positions, max=self.max_sequence_length - 1)
        positions = positions.unsqueeze(0).expand(bsz, -1)

        x = self.token_embedding(token_ids) + self.position_embedding(positions)
        x = self.dropout(x)
        for layer in self.layers:
            x = layer(x, position_offset=int(max(0, position_offset)))

        logits = self.lm_head(self.final_norm(x)) * float(self.output_logit_scale)
        return (logits, None) if return_memory else logits

    @torch.no_grad()
    def generate(
        self,
        seed_tokens: Sequence[int] | torch.Tensor,
        max_new_tokens: int,
        temperature: float = 0.9,
        top_p: float = 0.95,
        top_k: int = 50,
        repetition_penalty: float = 1.1,
        repetition_window: int = 64,
        min_tokens_to_keep: int = 3,
        seed_onset_times: Sequence[float] | torch.Tensor | None = None,
        step_seconds: float = 0.1,
        token_id_to_events: Any = None,
    ) -> List[int]:
        self.eval()
        device = next(self.parameters()).device

        tokens = self._to_seed_tensor(seed_tokens, device=device)
        if seed_onset_times is None:
            onsets = (
                torch.arange(tokens.shape[1], device=device, dtype=torch.float32)
                * float(max(1e-4, step_seconds))
            ).unsqueeze(0)
        else:
            if isinstance(seed_onset_times, torch.Tensor):
                on = seed_onset_times
                if on.ndim == 1:
                    on = on.unsqueeze(0)
                onsets = on.to(device=device, dtype=torch.float32)
            else:
                onsets = torch.tensor([float(v) for v in seed_onset_times], dtype=torch.float32, device=device).unsqueeze(0)

        if onsets.shape != tokens.shape:
            raise ValueError("seed_onset_times shape must match seed token shape")

        final_top1_probs: List[float] = []
        raw_top1_probs: List[float] = []
        candidate_counts: List[int] = []

        for _ in range(int(max_new_tokens)):
            context_tokens = tokens[:, -self.max_sequence_length:]
            context_onsets = onsets[:, -self.max_sequence_length:]
            context_offset = max(0, int(tokens.shape[1] - context_tokens.shape[1]))

            logits, _ = self.forward(
                token_ids=context_tokens,
                onset_times=context_onsets,
                memory=None,
                return_memory=True,
                position_offset=context_offset,
            )

            next_slot = self._triplet_slot(int(tokens.shape[1]))
            masked_logits = self._mask_logits_to_triplet_slot(logits[:, -1, :], next_slot)
            next_token, diagnostics = sample_next_token(
                logits=masked_logits,
                context_tokens=context_tokens,
                temperature=temperature,
                top_p=top_p,
                top_k=top_k,
                repetition_penalty=repetition_penalty,
                recent_window=repetition_window,
                min_tokens_to_keep=max(4, min_tokens_to_keep),
            )

            final_top1_probs.extend([float(v) for v in diagnostics.final_top1_prob.detach().cpu().tolist()])
            raw_top1_probs.extend([float(v) for v in diagnostics.raw_top1_prob.detach().cpu().tolist()])
            candidate_counts.extend([int(v) for v in diagnostics.candidate_count.detach().cpu().tolist()])

            tokens = torch.cat([tokens, next_token], dim=1)
            slot = self._triplet_slot(int(tokens.shape[1] - 1))
            tok = int(next_token.view(-1)[0].item())
            delta = float(max(1e-4, step_seconds))
            if slot == 0:
                delta = self._delta_from_token_events(tok, token_id_to_events, step_seconds)
            next_onset = onsets[:, -1:] + (delta if slot == 0 else 0.0)
            onsets = torch.cat([onsets, next_onset], dim=1)

        self.last_generation_stats = {
            "steps": int(max_new_tokens),
            "mean_final_top1_prob": float(sum(final_top1_probs) / max(1, len(final_top1_probs))),
            "mean_raw_top1_prob": float(sum(raw_top1_probs) / max(1, len(raw_top1_probs))),
            "mean_candidate_count": float(sum(candidate_counts) / max(1, len(candidate_counts))),
        }
        return [int(t) for t in tokens[0].tolist()]


def _filtered_config_payload(payload: Dict[str, Any]) -> Dict[str, Any]:
    allowed = {field.name for field in fields(VariantEConfig)}
    return {key: value for key, value in dict(payload).items() if key in allowed}


def _load_checkpoint_payload() -> Tuple[Dict[str, Any], Dict[str, Any]]:
    state_path = MODEL_DIR / "latest_state.pt"
    weights_path = MODEL_DIR / "latest.safetensors"

    if state_path.exists():
        state = torch.load(state_path, map_location="cpu")
        if not isinstance(state, dict):
            raise RuntimeError(f"Unsupported checkpoint state: {state_path}")
        return dict(state.get("model_config") or {}), dict(state.get("data_config") or {})

    if weights_path.exists():
        with safetensors_safe_open(str(weights_path), framework="pt", device="cpu") as f:
            metadata = f.metadata() or {}
        model_raw = metadata.get("model_config", "{}")
        data_raw = metadata.get("data_config", "{}")
        model_payload = json.loads(model_raw) if isinstance(model_raw, str) and model_raw.strip() else {}
        data_payload = json.loads(data_raw) if isinstance(data_raw, str) and data_raw.strip() else {}
        return model_payload, data_payload

    raise FileNotFoundError("No checkpoint state or safetensors weights found in the model directory.")


if "MIDI_INPUT_FILES" not in globals() or not MIDI_INPUT_FILES:
    raise RuntimeError("Run Cell 3 first so MIDI_INPUT_FILES is populated.")

model_payload, data_payload = _load_checkpoint_payload()
tokenizer_path = TOKENIZER_DIR / "custom_tokenizer.json"
if not tokenizer_path.exists():
    tokenizer_path = TOKENIZER_DIR / "tokenizer.json"
if not tokenizer_path.exists():
    raise FileNotFoundError("No tokenizer JSON found in the tokenizer directory.")

tokenizer = CustomDeltaTokenizer.load(str(tokenizer_path))
cfg_payload = _filtered_config_payload(model_payload)
config = VariantEConfig(**cfg_payload)
config = VariantEConfig(**{**config.__dict__, "vocab_size": int(tokenizer.vocab_size)})
model = VariantEModel(config)

weights_path = MODEL_DIR / "latest.safetensors"
if not weights_path.exists():
    raise FileNotFoundError(f"Model weights not found: {weights_path}")

state_dict = safetensors_load_file(str(weights_path), device="cpu")
state_dict = _strip_dataparallel_prefix(state_dict)
missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=False)
if missing_keys or unexpected_keys:
    missing_preview = ", ".join(missing_keys[:8]) if missing_keys else "none"
    unexpected_preview = ", ".join(unexpected_keys[:8]) if unexpected_keys else "none"
    if not GDN_AVAILABLE:
        raise RuntimeError(
            "Checkpoint/model mismatch detected and flash-linear-attention is unavailable. "
            "This GDN checkpoint likely needs true GatedDeltaNet kernels. "
            "Install `flash-linear-attention` in Colab and rerun from Cell 2. "
            f"missing={len(missing_keys)} ({missing_preview}) | "
            f"unexpected={len(unexpected_keys)} ({unexpected_preview})"
        )
    raise RuntimeError(
        "Checkpoint/model mismatch detected. "
        f"missing={len(missing_keys)} ({missing_preview}) | "
        f"unexpected={len(unexpected_keys)} ({unexpected_preview})"
    )

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(DEVICE)
model.eval()

trained_seed_length = max(1, int(data_payload.get("seed_length", 512) or 512))
trained_continuation_length = int(data_payload.get("continuation_length", 0) or 0)
if trained_continuation_length <= 0:
    max_sequence_length = int(data_payload.get("max_sequence_length", 0) or 0)
    trained_continuation_length = max(1, max_sequence_length - trained_seed_length) if max_sequence_length > 0 else 2048

SEED_LENGTH = max(4, int(os.environ.get("IBP_SEED_LENGTH", str(trained_seed_length))))
MAX_NEW_TOKENS = max(1, int(os.environ.get("IBP_MAX_NEW_TOKENS", str(trained_continuation_length))))
TEMPERATURE = max(0.1, float(os.environ.get("IBP_TEMPERATURE", "0.90")))
TOP_P = min(1.0, max(0.0, float(os.environ.get("IBP_TOP_P", "0.95"))) )
TOP_K = max(8, int(os.environ.get("IBP_TOP_K", "64")))
REPETITION_PENALTY = max(1.0, float(os.environ.get("IBP_REPETITION_PENALTY", "1.10")))
REPETITION_WINDOW = max(16, int(os.environ.get("IBP_REPETITION_WINDOW", "64")))
MIN_TOKENS_TO_KEEP = max(3, int(os.environ.get("IBP_MIN_TOKENS_TO_KEEP", "4")))
STEP_SECONDS = float(data_payload.get("time_feature_fallback_step_seconds", 0.1) or 0.1)

BATCH_OUTPUT_DIR = OUTPUT_DIR / "batch_continuations"
BATCH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Generation device: {DEVICE}")
print(f"Tokenizer: {tokenizer_path.name} | vocab={tokenizer.vocab_size}")
print(f"GDN kernel mode: {'flash-linear-attention' if GDN_AVAILABLE else 'fallback'}")
print(f"Songs queued: {len(MIDI_INPUT_FILES)}")
print(f"Seed length: {SEED_LENGTH} | Max new tokens: {MAX_NEW_TOKENS}")


def _safe_stem(path: Path) -> str:
    stem = re.sub(r"[^A-Za-z0-9._-]+", "_", path.stem).strip("._")
    return stem or "song"


def _trim_seed(token_ids: List[int], onset_times: List[float], seed_length: int, event_size: int) -> Tuple[List[int], List[float]]:
    take = min(len(token_ids), int(seed_length))
    aligned = take - (take % event_size)
    take = aligned if aligned > 0 else min(len(token_ids), event_size)
    return token_ids[-take:], onset_times[-take:]


BATCH_RESULTS: List[Dict[str, Any]] = []
total_files = len(MIDI_INPUT_FILES)
batch_start = time.time()

for index, midi_path in enumerate(MIDI_INPUT_FILES, start=1):
    song_start = time.time()
    safe_name = _safe_stem(midi_path)
    prefix = f"{index:03d}_{safe_name}"

    out_midi = BATCH_OUTPUT_DIR / f"{prefix}_continuation.mid"
    seed_audio = BATCH_OUTPUT_DIR / f"{prefix}_seed.wav"
    out_audio = BATCH_OUTPUT_DIR / f"{prefix}_continuation.wav"
    compare_png = BATCH_OUTPUT_DIR / f"{prefix}_comparison.png"

    print(f"\n[{index}/{total_files}] Starting: {midi_path.name}")
    try:
        seed_tokens, seed_onsets, _ = tokenizer.encode_with_time_features(midi_path)
        if not seed_tokens:
            raise RuntimeError("Tokenizer produced no seed tokens for this MIDI file.")

        seed_tokens, seed_onsets = _trim_seed(
            seed_tokens,
            seed_onsets,
            seed_length=SEED_LENGTH,
            event_size=max(1, int(getattr(tokenizer, "event_size", 4) or 4)),
        )

        with torch.inference_mode():
            generated_tokens = model.generate(
                seed_tokens=seed_tokens,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                top_k=TOP_K,
                repetition_penalty=REPETITION_PENALTY,
                repetition_window=REPETITION_WINDOW,
                min_tokens_to_keep=MIN_TOKENS_TO_KEEP,
                seed_onset_times=seed_onsets,
                step_seconds=STEP_SECONDS,
                token_id_to_events=tokenizer.decode_token_id_events,
            )

        tokenizer.decode(generated_tokens, out_midi)
        render_midi_audio(midi_path, seed_audio)
        render_midi_audio(out_midi, out_audio)
        compare_pianorolls(midi_path, out_midi, save_path=compare_png)

        song_seconds = float(time.time() - song_start)
        stats = dict(getattr(model, "last_generation_stats", {}))

        print(f"[{index}/{total_files}] Finished: {midi_path.name} ({song_seconds:.1f}s)")
        print(f"  output midi: {out_midi}")
        print(f"  seed audio: {seed_audio}")
        print(f"  continuation audio: {out_audio}")
        print(f"  comparison png: {compare_png}")
        print(f"  generation stats: {stats}")

        if display is not None and Audio is not None and Image is not None:
            display(Audio(filename=str(seed_audio)))
            display(Audio(filename=str(out_audio)))
            display(Image(filename=str(compare_png)))
        else:
            print("  IPython display unavailable; skipping inline audio/image display.")

        BATCH_RESULTS.append(
            {
                "index": index,
                "input_midi": str(midi_path),
                "output_midi": str(out_midi),
                "seed_audio": str(seed_audio),
                "continuation_audio": str(out_audio),
                "comparison_png": str(compare_png),
                "status": "ok",
                "elapsed_seconds": round(song_seconds, 3),
                "generated_tokens": int(len(generated_tokens)),
                "stats": stats,
            }
        )

    except Exception as exc:
        song_seconds = float(time.time() - song_start)
        print(f"[{index}/{total_files}] FAILED: {midi_path.name} ({song_seconds:.1f}s)")
        print(f"  error: {exc}")
        BATCH_RESULTS.append(
            {
                "index": index,
                "input_midi": str(midi_path),
                "status": "error",
                "elapsed_seconds": round(song_seconds, 3),
                "error": str(exc),
            }
        )


success_count = sum(1 for item in BATCH_RESULTS if item.get("status") == "ok")
fail_count = len(BATCH_RESULTS) - success_count
elapsed_total = float(time.time() - batch_start)

print("\nBatch continuation complete.")
print(f"Succeeded: {success_count} | Failed: {fail_count} | Total: {len(BATCH_RESULTS)}")
print(f"Elapsed: {elapsed_total:.1f}s")
print(f"Outputs: {BATCH_OUTPUT_DIR}")

if fail_count:
    print("Failed files:")
    for item in BATCH_RESULTS:
        if item.get("status") != "ok":
            print(f"  - {Path(item['input_midi']).name}: {item.get('error', 'unknown error')}")

In [ ]:
import json
import zipfile
from pathlib import Path

if "BATCH_RESULTS" not in globals():
    raise RuntimeError("Run Cell 6 first to generate continuations.")

batch_dir = OUTPUT_DIR / "batch_continuations"
batch_dir.mkdir(parents=True, exist_ok=True)

manifest_path = batch_dir / "batch_manifest.json"
manifest_path.write_text(json.dumps(BATCH_RESULTS, indent=2), encoding="utf-8")

ok_items = [item for item in BATCH_RESULTS if item.get("status") == "ok"]
err_items = [item for item in BATCH_RESULTS if item.get("status") != "ok"]

print(f"Manifest written: {manifest_path}")
print(f"Successful songs: {len(ok_items)}")
print(f"Failed songs: {len(err_items)}")

if err_items:
    print("Failures:")
    for item in err_items:
        print(f"  - {Path(item['input_midi']).name}: {item.get('error', 'unknown error')}")

ZIP_OUTPUTS = False
if ZIP_OUTPUTS:
    zip_path = OUTPUT_DIR / "batch_continuations.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for file_path in sorted(batch_dir.rglob("*")):
            if file_path.is_file():
                zf.write(file_path, arcname=str(file_path.relative_to(batch_dir)))
    print(f"Created zip: {zip_path}")